In [2]:
# SCRIPT DE ESTIMACIÓN Y VALIDACIÓN DE PENDIENTES NETAS (SPEI-6 vs SPI-6)
# Driscoll-Kraay robust linear combinations

import pandas as pd
import numpy as np
import scipy.stats as stats
from linearmodels.panel import RandomEffects
from patsy import dmatrices

# 1. Cargar datos
df = pd.read_csv('/home/hjvargaso/proyectos/tesis_sequia_2/data/processed/datos_mensuales_con_spei.csv')
df['fecha'] = pd.to_datetime(df['fecha'])

# OJO: Necesitamos calcular el SPI-6 para el contraste. 
# Si ya lo tiene en su base, cárguelo, de lo contrario lo aproximamos 
# o asumimos que ya existe en df como 'spi_6'. 
# Si no está, por favor verifique que su base lo contenga.
# Para este script, asumiremos que existe 'spi_6' en su panel.

cluster_mapping = {
    'MINGA': 1, 'SJBA': 1, 'VILL': 1, 'CAAZ': 1, 'COVIE': 1, 'CMEZA': 1, 'ENCAR': 1,
    'MCAL': 2, 'SEST': 2, 'SPED': 2, 'GBRU': 2, 'CONC': 2, 'PCOL': 2, 'LUQUE': 2, 'PJC': 2, 'PTOC': 2, 'PILAR': 2, 'SGUA': 2
}
df['cluster'] = df['estacion_id'].map(cluster_mapping)
df['tiempo'] = (df['fecha'].dt.year - 2000) * 12 + df['fecha'].dt.month
df['estacion_climatica'] = np.where(df['fecha'].dt.month.isin([12, 1, 2]), 'verano',
                           np.where(df['fecha'].dt.month.isin([3, 4, 5]), 'otono',
                           np.where(df['fecha'].dt.month.isin([9, 10, 11]), 'primavera', 'invierno')))
df['region_geografica'] = df['cluster'].astype(str)
df_panel = df.set_index(['estacion_id', 'fecha'])

# Definición de las 8 combinaciones lineales (Pesos w en el vector de 16 parámetros)
# El vector tiene la estructura del modelo dmatrices
scenarios = {
    'C2_invierno':  {'region': 'Clúster 2 (Centro-Chaco)', 'estacion': 'Invierno',  'weights': [8]},
    'C2_otono':     {'region': 'Clúster 2 (Centro-Chaco)', 'estacion': 'Otoño',     'weights': [8, 9]},
    'C2_primavera': {'region': 'Clúster 2 (Centro-Chaco)', 'estacion': 'Primavera', 'weights': [8, 10]},
    'C2_verano':    {'region': 'Clúster 2 (Centro-Chaco)', 'estacion': 'Verano',    'weights': [8, 11]},
    'C1_invierno':  {'region': 'Clúster 1 (Sur-Este)',     'estacion': 'Invierno',  'weights': [8, 12]},
    'C1_otono':     {'region': 'Clúster 1 (Sur-Este)',     'estacion': 'Otoño',     'weights': [8, 9, 12, 13]},
    'C1_primavera': {'region': 'Clúster 1 (Sur-Este)',     'estacion': 'Primavera', 'weights': [8, 10, 12, 14]},
    'C1_verano':    {'region': 'Clúster 1 (Sur-Este)',     'estacion': 'Verano',    'weights': [8, 11, 12, 15]}
}

def estimate_net_slopes(dep_var):
    formula = f"{dep_var} ~ 1 + tiempo * C(estacion_climatica, Treatment('invierno')) * C(region_geografica, Treatment('2'))"
    y, X = dmatrices(formula, df_panel, return_type='dataframe')
    model = RandomEffects(y, X).fit(cov_type='kernel', kernel='bartlett')
    
    coefs = model.params
    cov_matrix = model.cov
    
    results = []
    for sc_name, sc_info in scenarios.items():
        # Crear vector de pesos w
        w = np.zeros(len(coefs))
        for idx in sc_info['weights']:
            w[idx] = 1.0
            
        # Calcular pendiente neta y su error estándar robusto: SE = sqrt(w' * V * w)
        slope_net = np.dot(w, coefs)
        se_net = np.sqrt(np.dot(np.dot(w, cov_matrix), w))
        
        # Calcular t-stat y p-valor (distribución normal asintótica)
        t_stat = slope_net / se_net
        p_val = 2 * (1 - stats.norm.cdf(abs(t_stat)))
        
        results.append({
            'Clúster': sc_info['region'],
            'Estación': sc_info['estacion'],
            f'slope_{dep_var}': slope_net,
            f'slope_anual_{dep_var}': slope_net * 12, # Anualizada
            f'se_{dep_var}': se_net,
            f'p_val_{dep_var}': p_val
        })
        
    return pd.DataFrame(results)

# Ejecutar estimación para ambos modelos
try:
    df_spei = estimate_net_slopes('spei_6')
    
    # Intentar calcular para SPI si la columna existe
    if 'spi_6' in df_panel.columns:
        df_spi = estimate_net_slopes('spi_6')
        df_merged = pd.merge(df_spei, df_spi, on=['Clúster', 'Estación'])
        df_merged['Diferencia_Anual'] = df_merged['slope_anual_spei_6'] - df_merged['slope_anual_spi_6']
        print("\n--- CÁLCULO DE PENDIENTES NETAS Y CONTRASTE COMPLETADO ---")
        print(df_merged[['Clúster', 'Estación', 'slope_anual_spei_6', 'p_val_spei_6', 'slope_anual_spi_6', 'p_val_spi_6', 'Diferencia_Anual']])
    else:
        print("\n--- CÁLCULO COMPLETADO ÚNICAMENTE PARA SPEI-6 (Columna 'spi_6' no encontrada) ---")
        print(df_spei[['Clúster', 'Estación', 'slope_anual_spei_6', 'p_val_spei_6']])
except Exception as e:
    print(f"Error durante la ejecución: {e}")


--- CÁLCULO COMPLETADO ÚNICAMENTE PARA SPEI-6 (Columna 'spi_6' no encontrada) ---
                    Clúster   Estación  slope_anual_spei_6  p_val_spei_6
0  Clúster 2 (Centro-Chaco)   Invierno            0.014530      0.314903
1  Clúster 2 (Centro-Chaco)      Otoño            0.011873      0.501451
2  Clúster 2 (Centro-Chaco)  Primavera           -0.013345      0.404654
3  Clúster 2 (Centro-Chaco)     Verano           -0.012029      0.472964
4      Clúster 1 (Sur-Este)   Invierno            0.012040      0.570517
5      Clúster 1 (Sur-Este)      Otoño           -0.008244      0.692486
6      Clúster 1 (Sur-Este)  Primavera           -0.011867      0.531763
7      Clúster 1 (Sur-Este)     Verano           -0.036599      0.108589
